# Выборка ambientCG для пилота LoRA

CPU. Точные дубли, все кандидаты dHash и буквенные варианты одного ID объединяются транзитивно. Из группы берётся одна карта. Консервативное правило может объединять разные поверхности. Подписи сохранены из зеркала.

In [1]:
from pathlib import Path
from io import BytesIO
from hashlib import sha256
from itertools import combinations
import json
import re
import numpy as np
import pandas as pd
from PIL import Image

repo = "dream-textures/textures-color-1k"
revision = "33c49b76fcf3a18b0d521d8e760c88d49f3e47bc"
from huggingface_hub import hf_hub_download
import pyarrow.parquet as pq

parquet = hf_hub_download(repo, "data/train-00000-of-00001-5049fdc57d654dc7.parquet",
                          revision=revision, repo_type="dataset")
data = (row for batch in pq.ParquetFile(parquet).iter_batches(batch_size=16)
        for row in batch.to_pylist())
rows = []
rejected = []
for i, row in enumerate(data):
    asset = row["image"]
    mid = Path(asset["path"]).stem
    try:
        im = Image.open(BytesIO(asset["bytes"]))
        im.load()
    except (OSError, ValueError) as error:
        rejected.append(dict(row=i, material_id=mid, reason=type(error).__name__))
        continue
    with im:
        rgb = im.convert("RGB")
        gray = rgb.convert("L")
        lo, hi = np.percentile(np.asarray(gray), [1, 99])
        thumb = np.asarray(gray.resize((9, 8), Image.Resampling.LANCZOS))
        dhash = np.packbits(thumb[:, 1:] > thumb[:, :-1]).tobytes().hex()
        rows.append(dict(row=i, material_id=mid, category=re.split(r"\d", mid, 1)[0],
                         caption=row["text"], width=im.width, height=im.height, mode=im.mode,
                         flat=bool(lo == hi), sha256_rgb=sha256(rgb.tobytes()).hexdigest(), dhash=dhash))
frame = pd.DataFrame(rows)
frame.to_csv("catalog_audit.csv", index=False)
pairs = []
for a, b in combinations(rows, 2):
    exact = (a["width"], a["height"], a["sha256_rgb"]) == (b["width"], b["height"], b["sha256_rgb"])
    distance = (int(a["dhash"], 16) ^ int(b["dhash"], 16)).bit_count()
    if exact or (not a["flat"] and not b["flat"] and distance <= 4):
        pairs.append([a["material_id"], b["material_id"], distance, exact])
pd.DataFrame(pairs, columns=["left", "right", "distance", "exact"]).to_csv("pairs.csv", index=False)
pd.DataFrame(rejected, columns=["row", "material_id", "reason"]).to_csv("rejected.csv", index=False)
summary = dict(rejected=len(rejected), rows=len(rows), categories=int(frame.category.nunique()), flat=int(frame.flat.sum()),
               repeated_ids=int(frame.material_id.duplicated().sum()), modes=frame["mode"].value_counts().to_dict(),
               exact_pairs=sum(p[3] for p in pairs), candidate_pairs=sum(not p[3] for p in pairs),
               categories_by_size=frame.category.value_counts().to_dict())


In [2]:
from zipfile import ZipFile

allowed = {"Asphalt","Bricks","Concrete","Fabric","Gravel","Ground","Leather","Marble",
           "Metal","PavingStones","Planks","Rock","Tiles","Wood","WoodFloor"}
parent = {r["material_id"]: r["material_id"] for r in rows}

def root(key):
    while parent[key] != key:
        parent[key] = parent[parent[key]]
        key = parent[key]
    return key

def join(a, b):
    x, y = sorted([root(a), root(b)])
    parent[y] = x

families = {}
for mid in sorted(parent):
    family = re.sub(r"(?<=\d)[A-Za-z]+$", "", mid)
    join(mid, families.setdefault(family, mid))
for left, right, _, _ in pairs:
    join(left, right)

def rank(key):
    return sha256(f"42/{key}".encode()).digest()

buckets = {}
used = set()
for r in sorted(rows, key=lambda r: rank(r["material_id"])):
    mid = r["material_id"]
    if (r["category"] not in allowed or r["flat"] or not r["caption"].strip()
            or (r["width"], r["height"]) != (512, 512) or root(mid) in used):
        continue
    used.add(root(mid))
    buckets.setdefault(r["category"], []).append(r)
selected = []
for i in range(max(map(len, buckets.values()))):
    for cat in sorted(buckets):
        if i < len(buckets[cat]):
            selected.append(buckets[cat][i])
selected = selected[:400]
assert len(selected) == len({root(r["material_id"]) for r in selected}) == 400

manifest = []
for cat in sorted(buckets):
    members = sorted((r for r in selected if r["category"] == cat), key=lambda r: rank(root(r["material_id"])))
    n = max(1, round(len(members)/10))
    assert len(members) > 2*n
    for i, r in enumerate(members):
        mid = r["material_id"]
        split = "validation" if i < n else "test" if i < 2*n else "train"
        manifest.append(dict(material_id=mid, group_id=root(mid), category=cat, caption=r["caption"],
                             color=f"{split}/{mid}.png", split=split, sha256_rgb=r["sha256_rgb"],
                             source_url=f"https://ambientcg.com/view?id={mid}", license="CC0",
                             dataset_revision=revision))
content = "".join(json.dumps(item, ensure_ascii=False)+"\n" for item in manifest).encode()
Path("manifest.jsonl").write_bytes(content)
wanted = {m["material_id"]: m for m in manifest}
written = 0
with ZipFile("ambientcg-400.zip", "w") as archive:
    archive.writestr("manifest.jsonl", content)
    for batch in pq.ParquetFile(parquet).iter_batches(batch_size=16):
        for entry in batch.to_pylist():
            asset = entry["image"]
            item = wanted.get(Path(asset["path"]).stem)
            if item is None:
                continue
            with Image.open(BytesIO(asset["bytes"])) as im:
                rgb = im.convert("RGB")
                assert sha256(rgb.tobytes()).hexdigest() == item["sha256_rgb"]
                buf = BytesIO()
                rgb.save(buf, format="PNG")
                archive.writestr(item["color"], buf.getvalue())
            written += 1
assert written == 400
summary = dict(images=written, categories=len(buckets), known_pairs=len(pairs),
               splits=pd.Series([m["split"] for m in manifest]).value_counts().to_dict(),
               manifest_sha256=sha256(content).hexdigest())
Path("split_summary.json").write_text(json.dumps(summary))
print(json.dumps(summary, indent=2))

{
  "images": 400,
  "categories": 15,
  "known_pairs": 205,
  "splits": {
    "train": 312,
    "validation": 44,
    "test": 44
  },
  "manifest_sha256": "8fac9bd943d6c045d8f504c888af438e33c56857ad24764eade15e8d26d188d8"
}
